In [1]:
print(1)

1


In [2]:
import anndata as ad
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [3]:
import matplotlib.pyplot as plt

In [4]:
from sklearn.decomposition import PCA

In [5]:
import dilimap as dmap

In [6]:
import pickle

# Download_data

In [7]:
l1000_phase1_path = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/l1000_phase1/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/'
l1000_phase1_files = os.listdir(l1000_phase1_path)

l1000_phase2_path = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/l1000_phase2/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/'
l1000_phase2_files = os.listdir(l1000_phase2_path)

In [8]:
l1000_phase1 = []
for file in tqdm(l1000_phase1_files):
    l1000_phase1.append(ad.read_h5ad(l1000_phase1_path + file))

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_v

In [9]:
l1000_phase2 = []
for file in tqdm(l1000_phase2_files):
    l1000_phase2.append(ad.read_h5ad(l1000_phase2_path + file))

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_v

In [10]:
dili_train_path = '../../../data/dilimap_train_val/raw/adata_training_counts.h5ad'
dili_val_path = '../../../data/dilimap_train_val/raw/adata_validation_counts.h5ad'

dili_train = ad.read_h5ad(dili_train_path)
dili_val = ad.read_h5ad(dili_val_path)

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [11]:
with open("../../dili.pkl", "rb") as file:
    df_dili = pickle.load(file)

In [12]:
df_compounds_train = dili_train.obs[['COMPOUND', 'CONCENTRATION_UM', 'TIMEPOINT_HOURS', 'SPLIT']]
df_compounds_val = dili_val.obs[['COMPOUND', 'CONCENTRATION_UM', 'TIMEPOINT_HOURS', 'SPLIT']]

In [13]:
df_compounds = pd.concat([df_compounds_train, df_compounds_val])

In [14]:
df_compounds = df_compounds.merge(df_dili[['pert_id', 'pubchem_cid']], how='left', left_on='COMPOUND', right_on='pert_id')

In [15]:
df_compounds.loc[df_compounds['COMPOUND'] == 'DMSO', 'pubchem_cid'] = 679

In [16]:
df_compounds['pubchem_cid'] = df_compounds['pubchem_cid'].fillna(-666).astype(int).astype(str).astype('category').replace({'-666': None, -666: None})

/home/icb/olga.novitskaia/tmp/ipykernel_1203690/269742383.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_compounds['pubchem_cid'] = df_compounds['pubchem_cid'].fillna(-666).astype(int).astype(str).astype('category').replace({'-666': None, -666: None})


# Overlapping compounds

In [17]:
df_compounds_l1000 = pd.read_csv('../../df_compounds.csv')

In [18]:
DILI_TIME = 24.0  
overlapping_compounds1 = {}
for j, l1000_phase1_j in enumerate(l1000_phase1):
    l1000_24h = l1000_phase1_j.obs[l1000_phase1_j.obs['pert_time_h'] == DILI_TIME]
    overlapping_compounds1[j] = len(set(df_compounds['pubchem_cid']).intersection(set(l1000_24h['pubchem_cid'])))

In [19]:
overlapping_compounds2 = {}
for j, l1000_phase2_j in enumerate(l1000_phase2):
    l1000_24h = l1000_phase2_j.obs[l1000_phase2_j.obs['pert_time_h'] == DILI_TIME]
    overlapping_compounds2[j] = len(set(df_compounds['pubchem_cid']).intersection(set(l1000_24h['pubchem_cid'])))

In [20]:
sorted(overlapping_compounds1.items(), key=lambda item: item[1], reverse=True)[:10]

[(47, 165),
 (14, 164),
 (25, 150),
 (54, 133),
 (5, 107),
 (37, 98),
 (62, 94),
 (0, 90),
 (43, 90),
 (7, 77)]

In [21]:
sorted(overlapping_compounds2.items(), key=lambda item: item[1], reverse=True)[:10]

[(3, 128),
 (8, 128),
 (14, 128),
 (16, 128),
 (17, 128),
 (13, 125),
 (21, 125),
 (1, 4),
 (19, 4),
 (22, 4)]

In [22]:
def match_by_cid_and_closest_log_dose(dili, l1000_adata):
    l1000_obs = l1000_adata.obs.copy()
    l1000_obs['log_dose'] = np.log(l1000_obs['pert_dose_uM'])

    dili['log_dose'] = np.log(dili['CONCENTRATION_UM'])

    l1000_by_cid = {cid: grp for cid, grp in l1000_obs.groupby('pubchem_cid', observed=True)}

    matched_l1000_idx = []
    difference = []
    for _, row in dili.iterrows():
        l1000_grp = l1000_by_cid[row['pubchem_cid']]
        closest = (l1000_grp['log_dose'] - row['log_dose']).abs().idxmin()
        difference.append((l1000_grp['log_dose'] - row['log_dose']).abs().min())
        matched_l1000_idx.append(closest)

    dili['matched_l1000_idx'] = matched_l1000_idx
    dili['diff'] = difference
    return dili, l1000_obs

def return_embeddings(adata, dim=64,):
    adata_obs = adata.obs.copy()
    pca = PCA(n_components=dim, random_state=42)
    emb_logFC = pca.fit_transform(adata.layers['logFC'])
    
    pca = PCA(n_components=dim, random_state=42)
    emb_t = pca.fit_transform(adata.layers['t'])
    
    adata_obs['PCA.logFC'] = emb_logFC.tolist()
    adata_obs['PCA.t'] = emb_t.tolist()
    return adata_obs

In [23]:
to_check = sorted(overlapping_compounds1.items(), key=lambda item: item[1], reverse=True)[:10]
for item in to_check:
    l_id = item[0]
    compounds_overlapped = list(set(df_compounds['pubchem_cid']).intersection(l1000_phase1[l_id][l1000_phase1[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))
    df_compounds_filtered = df_compounds[df_compounds['pubchem_cid'].isin(compounds_overlapped)].copy().dropna()

    df_compounds_filtered = df_compounds_filtered[(df_compounds_filtered['CONCENTRATION_UM'] < 15) & (df_compounds_filtered['CONCENTRATION_UM'] > 5)].copy()
    
    
    l1000_phase1_no_duplicates = l1000_phase1[l_id][~l1000_phase1[l_id].obs.index.duplicated()]
    l1000_phase1_filtered = l1000_phase1_no_duplicates[l1000_phase1_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
    l1000_phase1_filtered_time = l1000_phase1_filtered[l1000_phase1_filtered.obs['pert_time_h'] == 24].copy()

    
    df_compounds_filtered_, _ = match_by_cid_and_closest_log_dose(df_compounds_filtered, l1000_phase1_filtered_time)
    l1000_phase1_filtered_time_obs = l1000_phase1_filtered_time.obs
    df_compounds_filtered_ = df_compounds_filtered_.merge(l1000_phase1_filtered_time_obs[['pert_dose_uM', 'pert_time_h']], how='left', left_on='matched_l1000_idx', right_index=True)
    
    print(l_id, len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()))
    

47 81
14 81
25 76
54 73
5 62
37 59
62 56
0 56
43 56
7 46


In [24]:
to_check = sorted(overlapping_compounds2.items(), key=lambda item: item[1], reverse=True)[:10]
for item in to_check:
    l_id = item[0]
    compounds_overlapped = list(set(df_compounds['pubchem_cid']).intersection(l1000_phase2[l_id][l1000_phase2[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))
    df_compounds_filtered = df_compounds[df_compounds['pubchem_cid'].isin(compounds_overlapped)].copy().dropna()

    df_compounds_filtered = df_compounds_filtered[(df_compounds_filtered['CONCENTRATION_UM'] < 15) & (df_compounds_filtered['CONCENTRATION_UM'] > 5)].copy()
    
    
    l1000_phase2_no_duplicates = l1000_phase2[l_id][~l1000_phase2[l_id].obs.index.duplicated()]
    l1000_phase2_filtered = l1000_phase2_no_duplicates[l1000_phase2_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
    l1000_phase2_filtered_time = l1000_phase2_filtered[l1000_phase2_filtered.obs['pert_time_h'] == 24].copy()

    
    df_compounds_filtered_, _ = match_by_cid_and_closest_log_dose(df_compounds_filtered, l1000_phase2_filtered_time)
    l1000_phase2_filtered_time_obs = l1000_phase2_filtered_time.obs
    df_compounds_filtered_ = df_compounds_filtered_.merge(l1000_phase2_filtered_time_obs[['pert_dose_uM', 'pert_time_h']], how='left', left_on='matched_l1000_idx', right_index=True)
    
    print(l_id, len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()))

3 59
8 59
14 59
16 59
17 59
13 59
21 59
1 1
19 1
22 1


# Processing

In [25]:
cols_l1000 = ['PCA.logFC', 
              'PCA.t',
              'cell_type',
              'pert_dose_uM', 
              'pert_time_h']

In [26]:
rename_l1000 = {'cell_type': 'l1000_cell_type',
                'pert_time_h': 'l1000_pert_time_h',
                'pert_dose_uM': 'l1000_pert_dose_uM'}

## Phase 1

In [27]:
def construct_data_filtered(query, reference, l_id):
    
    df_compounds = query.copy()
    compounds_overlapped = list(set(df_compounds['pubchem_cid']).intersection(reference[l_id][reference[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))
    df_compounds_filtered = df_compounds[df_compounds['pubchem_cid'].isin(compounds_overlapped)].copy().dropna()
    df_compounds_filtered = df_compounds_filtered[(df_compounds_filtered['CONCENTRATION_UM'] < 15) & (df_compounds_filtered['CONCENTRATION_UM'] > 5)].copy()
    
    reference_no_duplicates = reference[l_id][~reference[l_id].obs.index.duplicated()].copy()
    reference_filtered = reference_no_duplicates[reference_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
    reference_filtered_time = reference_filtered[reference_filtered.obs['pert_time_h'] == 24].copy()
    
    df_compounds_filtered_, _ = match_by_cid_and_closest_log_dose(df_compounds_filtered, reference_filtered_time)
    
    ## filter by prevailing dose
    reference_filtered_time_obs = reference_filtered_time.obs
    
    df_compounds_filtered_ = df_compounds_filtered_ = df_compounds_filtered_.merge(reference_filtered_time_obs[['pert_dose_uM', 'pert_time_h']], how='left', left_on='matched_l1000_idx', right_index=True)
    df_compounds_filtered_ = df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]
    df_compounds_filtered_ = df_compounds_filtered_.drop_duplicates(['COMPOUND', 'CONCENTRATION_UM'])

    return df_compounds_filtered_, reference_no_duplicates, reference_filtered, reference_filtered_time

In [28]:
def get_v1(df_compounds_filtered_, reference_filtered_time, dim=64):
    #V1
    
    df = reference_filtered_time[(reference_filtered_time.obs.index.isin(df_compounds_filtered_['matched_l1000_idx']))]
    df_emb = return_embeddings(df, dim)
    
    print('matched doses:', df_emb['pert_dose_uM'].unique())
    df_emb_dili = df_compounds_filtered_.merge(df_emb[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')
    df_emb_dili['version'] = 'v1'
    df_emb_dili['dim'] = dim
    return df_emb_dili

In [29]:
def get_v2(df_compounds_filtered_, reference_no_duplicates, dim=64):
    #V2
    df = reference_no_duplicates.copy()
    df_emb = return_embeddings(df, dim)
    
    df_emb_dili = df_compounds_filtered_.merge(df_emb[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')
    df_emb_dili['version'] = 'v2'
    df_emb_dili['dim'] = dim
    return df_emb_dili

In [30]:
def get_v3(df_compounds_filtered_, reference_no_duplicates, dim=64):
    df = reference_no_duplicates[(reference_no_duplicates.obs['pert_dose_uM'].isin([10]))&(reference_no_duplicates.obs['pert_time_h'].isin([24.]))].copy()
    df_emb = return_embeddings(df, dim)
    df_emb_dili = df_compounds_filtered_.merge(df_emb[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')
    df_emb_dili['version'] = 'v3'
    df_emb_dili['dim'] = dim
    return df_emb_dili

In [31]:
def get_v4(df_compounds_filtered_, reference_filtered, dim=64):
    df = reference_filtered.copy()
    df_emb = return_embeddings(df, dim)
    df_emb_dili = df_compounds_filtered_.merge(df_emb[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')
    df_emb_dili['version'] = 'v4'
    df_emb_dili['dim'] = dim
    return df_emb_dili

## Phase 1

In [32]:
columns = ['COMPOUND', 
       'CONCENTRATION_UM', 'TIMEPOINT_HOURS',
        'SPLIT',
        'pubchem_cid', 'matched_l1000_idx',
        'l1000_cell_type',
        'l1000_pert_dose_uM', 'l1000_pert_time_h', 
        'PCA.logFC', 'PCA.t', 'version', 'dim',
       'l1000_phase']

In [33]:
os.makedirs("strict_filtration", exist_ok=True)

In [34]:
to_check = sorted(overlapping_compounds1.items(), key=lambda item: item[1], reverse=True)[:10]

for item in tqdm(to_check):
    embeddings_phase1 = []
    l_id = item[0]
    df_compounds_filtered_, reference_no_duplicates, reference_filtered, reference_filtered_time = construct_data_filtered(df_compounds, l1000_phase1, l_id)
    if len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()) >= 50:
        print(len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()))
        for dim in [32, 64, 128]:
            try:
                embeddings_phase1.append(get_v1(df_compounds_filtered_, reference_filtered_time, dim=dim))
            except:
                pass
            try:
                embeddings_phase1.append(get_v2(df_compounds_filtered_, reference_no_duplicates, dim=dim))
            except:
                pass
            try:    
                embeddings_phase1.append(get_v3(df_compounds_filtered_, reference_no_duplicates, dim=dim))
            except:
                pass
            try:
                embeddings_phase1.append(get_v4(df_compounds_filtered_, reference_filtered, dim=dim))
            except:
                pass
        dili_emb = pd.concat(embeddings_phase1)
        dili_emb['l1000_phase'] = 1
        dili_emb = dili_emb[~dili_emb['PCA.t'].isna()]
        dili_emb = dili_emb[columns].reset_index(drop=True).copy()
        
        ct = l1000_phase1[l_id].obs['cell_type'].iloc[0]
        
        dili_emb.to_pickle('./strict_filtration/dili_PCA_emb_'+ ct + '_phase1' +'.pkl')

  0%|          | 0/10 [00:00<?, ?it/s]

81
matched doses: [10.]
matched doses: [10.]


 10%|█         | 1/10 [00:46<06:54, 46.03s/it]

81
matched doses: [10.]
matched doses: [10.]


 20%|██        | 2/10 [01:13<04:41, 35.20s/it]

...

 70%|███████   | 7/10 [03:57<01:34, 31.44s/it]

56
matched doses: [10.]


 80%|████████  | 8/10 [04:10<00:50, 25.36s/it]

56
matched doses: [10.]


100%|██████████| 10/10 [04:44<00:00, 28.41s/it]


## Phase 2

In [35]:
columns = ['COMPOUND', 
       'CONCENTRATION_UM', 'TIMEPOINT_HOURS',
        'SPLIT',
        'pubchem_cid', 'matched_l1000_idx',
           'l1000_cell_type',
        'l1000_pert_dose_uM', 'l1000_pert_time_h', 
        'PCA.logFC', 'PCA.t', 'version', 'dim',
       'l1000_phase']

In [36]:
to_check = sorted(overlapping_compounds2.items(), key=lambda item: item[1], reverse=True)[:10]

for item in tqdm(to_check):
    embeddings_phase2 = []
    l_id = item[0]
    df_compounds_filtered_, reference_no_duplicates, reference_filtered, reference_filtered_time = construct_data_filtered(df_compounds, l1000_phase2, l_id)
    if len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()) >= 50:
        print(len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()))
        for dim in [32, 64, 128]:
            try:
                embeddings_phase2.append(get_v1(df_compounds_filtered_, reference_filtered_time, dim=dim))
            except:
                pass
            try:
                embeddings_phase2.append(get_v2(df_compounds_filtered_, reference_no_duplicates, dim=dim))
            except:
                pass
            try:
                embeddings_phase2.append(get_v3(df_compounds_filtered_, reference_no_duplicates, dim=dim))
            except:
                pass
            try:
                embeddings_phase2.append(get_v4(df_compounds_filtered_, reference_filtered, dim=dim))
            except:
                pass
        dili_emb = pd.concat(embeddings_phase2)
        dili_emb['l1000_phase'] = 2
        dili_emb = dili_emb[~dili_emb['PCA.t'].isna()]
        dili_emb = dili_emb[columns].reset_index(drop=True).copy()

        ct = l1000_phase2[l_id].obs['cell_type'].iloc[0]
    
        dili_emb.to_pickle('./strict_filtration/dili_PCA_emb_'+ ct + '_phase2' +'.pkl')

  0%|          | 0/10 [00:00<?, ?it/s]

59
matched doses: [10.]


 10%|█         | 1/10 [00:17<02:37, 17.47s/it]

59
matched doses: [10.]


 20%|██        | 2/10 [00:33<02:13, 16.69s/it]

...

 50%|█████     | 5/10 [01:24<01:26, 17.25s/it]

59
matched doses: [10.]


 60%|██████    | 6/10 [01:47<01:17, 19.31s/it]

59
matched doses: [10.]


100%|██████████| 10/10 [02:15<00:00, 13.50s/it]
